# ACCEDER A MINIO Y BORRAR TODO LO QUE EXISTA EN EL BUCKET DE HOSPITAL


# Crear el Catalogo para la arquitectura Medallion
## 1. Crear el Catalogo con el que gestionaremos nuestra arquitectura

In [1]:
from pyspark.sql import SparkSession
from delta import *



spark = (SparkSession.builder 
    .appName("CrearArquitecturaMedallion") 
    .master("spark://spark-master:7077") 
    .config("spark.submit.deployMode", "client") 
         
    # Extensiones Delta 
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
         
    # Crea el catálogo como un DeltaCatalog
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
         
    .config("spark.sql.warehouse.dir", "s3a://hospital/")
    # estamos usando hive en memoria, el catalogo no persiste entre sesiones
         
    # Establecer la conexión a minio
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") 
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") 
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") 
    .config("spark.hadoop.fs.s3a.path.style.access", "true") 
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") 
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")   
    .config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1") 
    .config("spark.jars","/opt/spark/jars/hadoop-aws-3.4.0.jar, /opt/spark/jars/aws-java-sdk-bundle-2.23.19.jar, /opt/spark/jars/delta-spark_2.13-4.0.0.jar, /opt/spark/jars/delta-storage-4.0.0.jar, /opt/spark/jars/antlr4-runtime-4.13.1.jar, /opt/spark/jars/spark-sql-kafka-0-10_2.13-4.0.1.jar, /opt/spark/jars/kafka-clients-3.9.1.jar, /opt/spark/jars/mssql-jdbc-12.10.2.jre11.jar") 
    .getOrCreate())


print("SparkSession creada")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-93b0b8df-611b-41e7-a990-d3f9c8cf2adb;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala

SparkSession creada


## 2. Creamos la capa bronze

2.1  Creamos el esquema para guardar las ingestas de la capa bronze

2.2 Para manejar incrementalidad real (solo nuevos cambios) creamos una tabla checkpoint por cada Tabla a ingestar. Esta tabla permitirá guardar el Número de Secuencia de Registro (LSN, __$start_lsn) encargado de registrar el identificador único de cada transacción que realizó un cambio (inserción, actualización o eliminación) en la tabla.

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, BinaryType
from pyspark.sql import Row


bronze_table_checkpoint="s3a://hospital/bronze/pacientes_checkpoint"

# CREAR TABLA DE checkpoint SI NO EXISTE
schema_processed = StructType([
    StructField("last_lsn", BinaryType(), True)
])
if not DeltaTable.isDeltaTable(spark, bronze_table_checkpoint):
    checkpoint_df = spark.createDataFrame([], schema_processed)
    
    # Crear DataFrame con la fila null
    row_df = spark.createDataFrame([Row(last_lsn=None)], schema_processed)
    
    # Unir (append)
    checkpoint_df = checkpoint_df.union(row_df)
    # Guardar como tabla Delta en S3 y registrar en el catálogo
    checkpoint_df.write \
      .format("delta") \
      .mode("overwrite") \
      .save(bronze_table_checkpoint)


26/10/27 16:40:54 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


##  3. Creamos la capa  SILVER

3.1 Creamos el esquema para la ingesta en SILVER

3.2 Creamos la tabla patients para la ingesta en SILVER que tendrá los datos limpios

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
schema = StructType([
    StructField("paciente_id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("edad", IntegerType(), True),
    StructField("genero", StringType(), True),
    StructField("email", StringType(), True),
    StructField("historia_medica", StringType(), True),
])
silver_table="s3a://hospital/silver/pacientes"

if not DeltaTable.isDeltaTable(spark, silver_table):
    silver_df = spark.createDataFrame([], schema)
    
    # Guardar como tabla Delta en S3 
    silver_df.write \
      .format("delta") \
      .mode("overwrite") \
      .save(silver_table)

    silver_df.printSchema()



root
 |-- paciente_id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- genero: string (nullable = true)
 |-- email: string (nullable = true)
 |-- historia_medica: string (nullable = true)



## 3. Creamos el esquema para la ingesta en GOLD

3.1 Creamos el esquema para la ingesta en GOLD

In [4]:
# crearíamos cualquier tabla que estimemos necesaria


# Creación de Tabla y Datos en SQL Server utiliza VS Code

## 1. Activar CDC en la Base de Datos

EXEC sys.sp_cdc_enable_db;

## 2. Creación de la Tabla Patients

CREATE TABLE pacientes (
    paciente_id INT PRIMARY KEY,
    nombre NVARCHAR(100) NOT NULL, 
    edad INT NOT NULL,
    genero CHAR(1) CHECK (genero IN ('M','F')),
    email NVARCHAR(100) NOT NULL,
    historia_medica NVARCHAR(255)
);

## 3. Activar Change Data Capture (CDC) sobre l
EXEC sys.sp_cdc_enable_table
    @source_schema = N'dbo',
    @source_name   = N'pacientes',
    @role_name  LL;
= NULL;

### 3.1 En caso de querer desactivarlo
EXEC sys.sp_cdc_disable_table
    @source_schema = 'dbo',
    @source_name = 'pacientes',
    @capture_instance = 'dboo## 4. Insertar datos simulados con email
INSERT INTO pacientes (paciente_id, nombre, edad, genero, email, historia_medica)
VALUES
(1, 'Juan Pérez', 45, 'M', 'juan.perez@example.com', 'Hipertensión'),
(2, 'María López', 52, 'F', 'maria.lopez@example.com', 'Diabetes tipo 2'),
(3, 'Carlos Ruiz', 38, 'M', 'carlos.ruiz@example.com', 'Sin antecedentes'),
(4, 'Ana Torres', 60, 'F', 'ana.torres@example.com', 'Arritmias'),
(5, 'Pedro Gómez', 70, 'M', 'pedro.gomez@example.com', 'Cardiopatía'),
(6, 'Laura Sánchez', 29, 'F', 'laura.sanchez@example.com', 'Sin antecedentes'),
(7, 'Miguel Ortega', 50, 'M', 'miguel.ortega@example.com', 'Hipertensión y obesidad');besidad');



# Ingesta inicial de los datos en Bronze y en silver

In [ ]:
Este script permite llenar la base de datos y se puede ejecutar con este comando
docker compose exec -T sqlserver /opt/mssql-tools18/bin/sqlcmd   -S localhost -U sa -P 'Password1234%' -C -f 65001 < scripts/init_sqlserver.sql


In [5]:
# leemos desde SQL Server

from delta.tables import DeltaTable
from pyspark.sql import functions as F

jdbc_url = (
    "jdbc:sqlserver://sqlserver:1433;"
    "databaseName=HOSPITAL;"
    "encrypt=true;"
    "trustServerCertificate=true;"
)
connection_properties = {
    "user": "sa", # tu usuario
    "password": "Password1234%", # tu password
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

pacientes_table = "(SELECT * FROM pacientes) paciente_alias"

# Leer desde la base de datos SQLServer
df_bronze = spark.read.jdbc(
    url=jdbc_url,
    table=pacientes_table,
    properties=connection_properties
)
df_bronze.show()

+-----------+-------------+----+------+--------------------+--------------------+
|paciente_id|       nombre|edad|genero|               email|     historia_medica|
+-----------+-------------+----+------+--------------------+--------------------+
|          1|   Juan Pérez|  45|     M|juan.perez@exampl...|        Hipertensión|
|          2|  María López|  52|     F|maria.lopez@examp...|     Diabetes tipo 2|
|          3|  Carlos Ruiz|  38|     M|carlos.ruiz@examp...|    Sin antecedentes|
|          4|   Ana Torres|  60|     F|ana.torres@exampl...|           Arritmias|
|          5|  Pedro Gómez|  70|     M|pedro.gomez@examp...|         Cardiopatía|
|          6|Laura Sánchez|  29|     F|laura.sanchez@exa...|    Sin antecedentes|
|          7|Miguel Ortega|  50|     M|miguel.ortega@exa...|Hipertensión y ob...|
+-----------+-------------+----+------+--------------------+--------------------+



In [6]:
bronze_table_path="s3a://hospital/bronze/pacientes"

df_bronze.write \
  .format("delta") \
  .mode("overwrite") \
  .save(bronze_table_path)




In [7]:
silver_table_path="s3a://hospital/silver/pacientes"

df_bronze.write \
  .format("delta") \
  .mode("overwrite") \
  .save(bronze_table_path)


# Filtrar filas con email no nulo y que contenga '@', edad entre 0 y 120, y genero F o M para crear una tabla correcta
silver_filtered_df = df_bronze.filter(
    ((df_bronze.email.isNotNull()) & (df_bronze.email.contains("@"))) & ((df_bronze.edad >= 0) & (df_bronze.edad <= 120)) & (df_bronze.genero.isin('F', 'M'))
)

silver_filtered_df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("mergeSchema", "true") \
  .save(silver_table_path)

silver_filtered_df.show()

26/10/27 17:11:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+-------------+----+------+--------------------+--------------------+
|paciente_id|       nombre|edad|genero|               email|     historia_medica|
+-----------+-------------+----+------+--------------------+--------------------+
|          1|   Juan Pérez|  45|     M|juan.perez@exampl...|        Hipertensión|
|          2|  María López|  52|     F|maria.lopez@examp...|     Diabetes tipo 2|
|          3|  Carlos Ruiz|  38|     M|carlos.ruiz@examp...|    Sin antecedentes|
|          4|   Ana Torres|  60|     F|ana.torres@exampl...|           Arritmias|
|          5|  Pedro Gómez|  70|     M|pedro.gomez@examp...|         Cardiopatía|
|          6|Laura Sánchez|  29|     F|laura.sanchez@exa...|    Sin antecedentes|
|          7|Miguel Ortega|  50|     M|miguel.ortega@exa...|Hipertensión y ob...|
+-----------+-------------+----+------+--------------------+--------------------+



# Cerrar la sesión para terminar el job

In [10]:
spark.stop()